# Linear Regression Trajectory (BiasWithMSETrajectory)

This notebook keeps trajectory assembly in-notebook and uses `BiasWithMSETrajectory`
for optimizer-based diagonal bias fitting.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from accelerate.test_utils.testing import get_backend
from lightning import Trainer
from lightning.pytorch.callbacks import EarlyStopping

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / 'src'))

from core.data import FullBatchDataModule
from core.bias import DiagMatrixRidgeBias
from core.estimators import BiasWithMSETrajectory
from core.utils import compute_Q_matrix, compute_beta_closed_form

torch.set_float32_matmul_precision('highest')
style_path = ROOT / 'clean_fig.mplstyle'
if style_path.exists():
    plt.style.use(style_path)

device, n_devices, _ = get_backend()
n_devices = 1
FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def mse_grad_full(X: torch.Tensor, y: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    return 2.0 * X.T @ (X @ theta - y) / X.shape[0]

def gd_trajectory(X: torch.Tensor, y: torch.Tensor, steps: int, eps: float) -> tuple[torch.Tensor, torch.Tensor]:
    theta = torch.zeros(X.shape[1], dtype=X.dtype, device=X.device)
    thetas = [theta.clone()]
    for _ in range(steps):
        grad_theta = mse_grad_full(X, y, theta)
        theta = theta - (eps / 2.0) * grad_theta
        thetas.append(theta.clone())
    thetas = torch.stack(thetas)
    targets = -torch.stack([mse_grad_full(X, y, t) for t in thetas])
    return thetas, targets

def stack_trajectory_points(
    theta_trajectories,
    target_trajectories,
    *,
    endpoints_only: bool = False,
    burn_in: int = 0,
) -> tuple[torch.Tensor, torch.Tensor]:
    if len(theta_trajectories) != len(target_trajectories):
        raise ValueError('theta_trajectories and target_trajectories must have the same length')
    theta_points = []
    target_points = []
    for theta_traj, target_traj in zip(theta_trajectories, target_trajectories):
        if theta_traj.shape != target_traj.shape:
            raise ValueError(f'shape mismatch: {theta_traj.shape} vs {target_traj.shape}')
        if endpoints_only:
            theta_points.append(theta_traj[-1:])
            target_points.append(target_traj[-1:])
        else:
            start = burn_in + 1
            if start >= theta_traj.shape[0]:
                raise ValueError('burn_in is too large for at least one trajectory')
            theta_points.append(theta_traj[start:])
            target_points.append(target_traj[start:])
    return torch.cat(theta_points, dim=0), torch.cat(target_points, dim=0)

def fit_diag_from_points(
    theta_points: torch.Tensor,
    target_points: torch.Tensor,
    *,
    q_init: torch.Tensor | None = None,
    lr: float = 1e-2,
    max_epochs: int = 5000,
    patience: int | None = 150,
) -> torch.Tensor:
    p = theta_points.shape[1]
    q_seed = None
    if q_init is not None:
        q_seed = q_init.detach().clone().reshape(-1)
        if q_seed.numel() != p:
            raise ValueError(f'q_init has {q_seed.numel()} entries, expected {p}')

    dm = FullBatchDataModule(theta_points, target_points, num_workers=0)
    estimator = BiasWithMSETrajectory(
        bias_model=DiagMatrixRidgeBias(dim=p, Q_init=q_seed),
        grad_factor=2.0,
        lr=lr,
        optimizer_cls=torch.optim.Adam,
    )

    callbacks = []
    if patience is not None:
        callbacks.append(
            EarlyStopping(
                monitor='train_bias/loss',
                mode='min',
                patience=patience,
                check_on_train_epoch_end=True,
            )
        )

    trainer = Trainer(
        max_epochs=max_epochs,
        logger=False,
        callbacks=callbacks,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
        log_every_n_steps=1,
        accelerator=device,
        devices=n_devices,
    )
    trainer.fit(estimator, dm)
    return estimator.bias_model.Q.detach().clone()

def fit_diag_from_trajectories(
    theta_trajectories,
    target_trajectories,
    *,
    endpoints_only: bool = False,
    burn_in: int = 0,
    q_init: torch.Tensor | None = None,
    lr: float = 1e-2,
    max_epochs: int = 5000,
    patience: int | None = 150,
) -> torch.Tensor:
    theta_points, target_points = stack_trajectory_points(
        theta_trajectories,
        target_trajectories,
        endpoints_only=endpoints_only,
        burn_in=burn_in,
    )
    return fit_diag_from_points(
        theta_points,
        target_points,
        q_init=q_init,
        lr=lr,
        max_epochs=max_epochs,
        patience=patience,
    )

def fit_single_trajectory(theta_trajectory, target_trajectory, *, burn_in: int = 0, **kwargs):
    return fit_diag_from_trajectories(
        [theta_trajectory],
        [target_trajectory],
        endpoints_only=False,
        burn_in=burn_in,
        **kwargs,
    )

def fit_multiple_endpoints(theta_trajectories, target_trajectories, **kwargs):
    return fit_diag_from_trajectories(
        theta_trajectories,
        target_trajectories,
        endpoints_only=True,
        burn_in=0,
        **kwargs,
    )

def relative_distance(est: torch.Tensor, theory: torch.Tensor) -> float:
    return (torch.linalg.norm(est - theory) / torch.linalg.norm(theory)).item()

def plot_figure3_variant(
    lam_est: torch.Tensor,
    Q_theory: torch.Tensor,
    X: torch.Tensor,
    y: torch.Tensor,
    theta_hat: torch.Tensor,
    theta_true: torch.Tensor,
    title: str,
    filename: str,
) -> None:
    Q_est = torch.diag(lam_est)
    theta_explicit = compute_beta_closed_form(X, y, Q_est)

    weights = torch.stack([theta_explicit, theta_hat, theta_true], dim=1).cpu().numpy()
    Q_est_np = Q_est.cpu().numpy()
    Q_theory_np = Q_theory.cpu().numpy()
    vmax_q = max(np.abs(Q_est_np).max(), np.abs(Q_theory_np).max())
    wmax = np.abs(weights).max()

    fig = plt.figure(figsize=(14, 5.8), constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.1, 0.8])

    ax0 = fig.add_subplot(gs[0, 0])
    im0 = ax0.imshow(Q_est_np, cmap='bwr', vmin=-vmax_q, vmax=vmax_q, aspect='equal')
    ax0.set_title(r'Estimated $\hat{\Lambda}$')
    ax0.set_xlabel(r'$p$')
    ax0.set_ylabel(r'$p$')

    ax1 = fig.add_subplot(gs[0, 1])
    ax1.imshow(Q_theory_np, cmap='bwr', vmin=-vmax_q, vmax=vmax_q, aspect='equal')
    ax1.set_title(r'Theoretical $\Lambda$')
    ax1.set_xlabel(r'$p$')
    fig.colorbar(im0, ax=[ax0, ax1], shrink=0.9, pad=0.02)

    ax2 = fig.add_subplot(gs[0, 2])
    im2 = ax2.imshow(weights, cmap='bwr', vmin=-wmax, vmax=wmax, aspect='equal')
    ax2.set_title(r'$\hat{\theta}_{\Lambda}$ vs $\hat{\theta}$ vs $\theta$')
    ax2.set_xticks([0, 1, 2])
    ax2.set_xticklabels([r'$\hat{\theta}_{\Lambda}$', r'$\hat{\theta}$', r'$\theta$'])
    ax2.set_ylabel(r'$p$')
    fig.colorbar(im2, ax=ax2, shrink=0.9, pad=0.02)

    fig.suptitle(title)
    fig.savefig(FIG_DIR / filename, bbox_inches='tight')
    plt.close(fig)


In [ ]:
SEED = 56
P = 10
N = 1000
EPS = 1e-2
T = 500
BURN_IN = 50
NUM_RUNS = 10

BIAS_LR = 1e-2
BIAS_SINGLE_MAX_EPOCHS = 5000
BIAS_ENDPOINT_MAX_EPOCHS = 500
BIAS_PATIENCE = 150

CURVE_LR = 2e-3
CURVE_STEPS_PER_PREFIX = 20
CURVE_GRAD_CLIP = 10.0

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.use_deterministic_algorithms(False)

X = torch.randn(N, P)

run_data = []
for r in range(NUM_RUNS):
    g = torch.Generator().manual_seed(SEED + 100 + r)
    beta_r = 3 * torch.randn(P, generator=g)
    y_r = X @ beta_r
    theta_r, target_r = gd_trajectory(X, y_r, T, EPS)
    run_data.append({'beta': beta_r, 'y': y_r, 'theta': theta_r, 'target': target_r})

single = run_data[0]
theta_single = single['theta']
target_single = single['target']
beta_single = single['beta']
y_single = single['y']

Q_theory_by_step = torch.stack([compute_Q_matrix(X, k, EPS) for k in range(1, T + 1)])
lambda_theory_by_step = torch.diagonal(Q_theory_by_step, dim1=-2, dim2=-1)

Q_theory_endpoint = Q_theory_by_step[T - 1]
lambda_theory_endpoint = lambda_theory_by_step[T - 1]

Q_theory_single = Q_theory_by_step[BURN_IN:T].mean(dim=0)
lambda_theory_single = torch.diag(Q_theory_single)

def single_prefix_theory_diag(m: int) -> torch.Tensor:
    # Average theory over exactly the steps used in the single-trajectory fit: burn_in+1..m
    return lambda_theory_by_step[BURN_IN:m].mean(dim=0)

print(f'single trajectory shape: {theta_single.shape}')
print(f'num trajectories: {len(run_data)}')


In [ ]:
# 1) Single trajectory special case (5000 epochs)
lambda_hat_single = fit_single_trajectory(
    theta_single,
    target_single,
    burn_in=BURN_IN,
    lr=BIAS_LR,
    max_epochs=BIAS_SINGLE_MAX_EPOCHS,
    patience=BIAS_PATIENCE,
)
dist_single = relative_distance(lambda_hat_single, lambda_theory_single)

plot_figure3_variant(
    lam_est=lambda_hat_single,
    Q_theory=Q_theory_single,
    X=X,
    y=y_single,
    theta_hat=theta_single[-1],
    theta_true=beta_single,
    title='Figure 3 Variant: Single-Trajectory Fit',
    filename='linear_regression_trajectory_fig3_single_trajectory.pdf',
)
print(f'single-trajectory distance: {dist_single:.6f}')


In [ ]:
# 2) Multiple endpoints special case (500 epochs)
theta_trajectories = [r['theta'] for r in run_data]
target_trajectories = [r['target'] for r in run_data]

lambda_hat_10 = fit_multiple_endpoints(
    theta_trajectories,
    target_trajectories,
    lr=BIAS_LR,
    max_epochs=BIAS_ENDPOINT_MAX_EPOCHS,
    patience=BIAS_PATIENCE,
)
dist_10 = relative_distance(lambda_hat_10, lambda_theory_endpoint)

plot_figure3_variant(
    lam_est=lambda_hat_10,
    Q_theory=Q_theory_endpoint,
    X=X,
    y=y_single,
    theta_hat=theta_single[-1],
    theta_true=beta_single,
    title='Figure 3 Variant: 10-Run Endpoint-Only Fit',
    filename='linear_regression_trajectory_fig3_10run_endpoint.pdf',
)
print(f'10-trajectory endpoint distance: {dist_10:.6f}')


In [ ]:
# 3) Distance curves (optimizer-based, oscillation-controlled)
step_counts = torch.arange(BURN_IN + 1, T + 1)
step_distances = torch.empty(step_counts.numel(), dtype=torch.float64)

q_step = torch.zeros(P, dtype=theta_single.dtype, device=theta_single.device, requires_grad=True)
opt_step = torch.optim.Adam([q_step], lr=CURVE_LR)

for i, m in enumerate(step_counts.tolist()):
    theta_points_m, target_points_m = stack_trajectory_points(
        [theta_single[: m + 1]],
        [target_single[: m + 1]],
        endpoints_only=False,
        burn_in=BURN_IN,
    )
    for _ in range(CURVE_STEPS_PER_PREFIX):
        opt_step.zero_grad(set_to_none=True)
        predicted_m = 2.0 * q_step.view(1, -1) * theta_points_m
        loss_m = torch.nn.functional.mse_loss(predicted_m, target_points_m, reduction='mean')
        loss_m.backward()
        torch.nn.utils.clip_grad_norm_([q_step], max_norm=CURVE_GRAD_CLIP)
        opt_step.step()
    step_distances[i] = relative_distance(q_step.detach(), single_prefix_theory_diag(m))

lam_endpoint_single = fit_multiple_endpoints(
    [theta_single],
    [target_single],
    lr=BIAS_LR,
    max_epochs=BIAS_ENDPOINT_MAX_EPOCHS,
    patience=BIAS_PATIENCE,
)
dist_endpoint_single = relative_distance(lam_endpoint_single, lambda_theory_endpoint)

run_counts = torch.arange(1, NUM_RUNS + 1)
run_distances = torch.empty(NUM_RUNS, dtype=torch.float64)

q_run = torch.zeros(P, dtype=theta_single.dtype, device=theta_single.device, requires_grad=True)
opt_run = torch.optim.Adam([q_run], lr=CURVE_LR)

for i, k in enumerate(run_counts.tolist()):
    theta_points_k, target_points_k = stack_trajectory_points(
        theta_trajectories[:k],
        target_trajectories[:k],
        endpoints_only=True,
        burn_in=0,
    )
    for _ in range(CURVE_STEPS_PER_PREFIX):
        opt_run.zero_grad(set_to_none=True)
        predicted_k = 2.0 * q_run.view(1, -1) * theta_points_k
        loss_k = torch.nn.functional.mse_loss(predicted_k, target_points_k, reduction='mean')
        loss_k.backward()
        torch.nn.utils.clip_grad_norm_([q_run], max_norm=CURVE_GRAD_CLIP)
        opt_run.step()
    run_distances[i] = relative_distance(q_run.detach(), lambda_theory_endpoint)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)
axes[0].semilogy(step_counts.numpy(), step_distances.numpy(), color='#1f77b4')
axes[0].axhline(dist_endpoint_single, color='red', linestyle='--')
axes[0].set_title('Single Trajectory')
axes[0].set_xlabel('Total Steps m (fit on burn-in+1..m)')
axes[0].set_ylabel('Relative Distance to Theory')
axes[0].grid(alpha=0.3, which='both')

axes[1].semilogy(run_counts.numpy(), run_distances.numpy(), marker='o', color='#d62728')
axes[1].set_title('Endpoint-Only Across Trajectories')
axes[1].set_xlabel('Number of Trajectories')
axes[1].set_ylabel('Relative Distance to Theory')
axes[1].grid(alpha=0.3, which='both')

fig.suptitle('Distance from Theoretical Regularizer')
fig.savefig(FIG_DIR / 'linear_regression_trajectory_distance_to_theory.pdf', bbox_inches='tight')
plt.close(fig)


In [ ]:
print(f'single-trajectory distance: {dist_single:.6f}')
print(f'single endpoint-only (1 trajectory) distance: {dist_endpoint_single:.6f}')
print(f'10-trajectory endpoint distance: {dist_10:.6f}')
print(FIG_DIR / 'linear_regression_trajectory_fig3_single_trajectory.pdf')
print(FIG_DIR / 'linear_regression_trajectory_fig3_10run_endpoint.pdf')
print(FIG_DIR / 'linear_regression_trajectory_distance_to_theory.pdf')
